# Regression Analysis Project: Predicting Freelancer Hourly Rate (USD)

## Mission
Predict a freelancer's **hourly rate (USD)** from their profile — demographics, country, skill, 
experience, rating, activity status, and client satisfaction — so platforms and freelancers can 
benchmark expected pricing. Non-generic use case — **not** a house-price prediction task.

## Dataset Description & Source
- **Name / file:** Global Freelancers (raw) dataset (`global_freelancers_raw.csv`)
- **Source:** tabular CSV export of freelancer profiles (Kaggle-style global freelance-marketplace dataset)
- **Volume:** 1,000 freelancer records, 12 raw columns
- **Variety:** 21 countries, 16 languages, 10 primary skills, numeric demographic/performance fields 
  (age, years of experience, rating, client satisfaction), plus several **genuinely messy real-world 
  fields**: `gender` has 10 inconsistent spellings/cases, `is_active` mixes `1/0`, `Y/N`, `yes/no`, 
  `True/False`, `hourly_rate (USD)` mixes plain numbers, `"$40"`, and `"USD 40"`, and 
  `client_satisfaction` mixes `"84%"` with bare `"92"`. Several columns also contain real missing 
  values (NaNs).
- **Target variable (regression):** `hourly_rate (USD)`

## What this notebook does
1. Loads the **raw, uncleaned** data and profiles exactly what is dirty about it
2. Cleans every messy column and handles all missing values (details in Section 1)
3. Two required visualizations that directly inform modeling: **feature distributions** and a 
   **correlation heatmap**
4. Feature engineering (drops, encodes, and justifies column choices)
5. Converts all remaining categorical/text columns to numeric
6. Standardizes numeric features
7. Implements **two different linear regression models via scikit-learn, both trained with gradient 
   descent** — (A) Stochastic Gradient Descent and (B) Mini-Batch Gradient Descent — and compares both 
   against a **Decision Tree Regressor** (tree) and a **Random Forest Regressor** (ensemble): 4 models total
8. States the criteria for choosing the best model, and saves it
9. Plots the best-fit line for the linear regression model over the dataset
10. Makes a prediction on one row of the actual test set
11. Provides a reusable prediction script for a brand-new freelancer profile (Task 2)


## 0. Setup — Imports


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)

RANDOM_STATE = 42
rng = np.random.RandomState(RANDOM_STATE)


ModuleNotFoundError: No module named 'numpy'

## Bridging This Notebook With the API (no duplicated logic)

The API (`summative/API/prediction.py`) is a **separate file**, as required by the project 
structure -- a notebook can't be called at runtime like a server, so it can't literally be "the" 
API. Instead, this notebook and the API both import the exact same cleaning/feature-engineering 
functions from **`preprocessing.py`** (inside `summative/API/`), so the logic exists in exactly one 
place, not as two hand-typed copies that could drift apart.

The cell below tries to fetch that file directly from GitHub and import it. If it can't reach GitHub 
(e.g. you haven't pushed the repo yet, or you're offline), the notebook falls back to an identical, 
hand-kept-in-sync local copy of the same functions defined further down, so the notebook still runs 
standalone.


In [ ]:
# --- Bridge: import the SAME cleaning/encoding code the API uses ---
# Replace <you>/<repo> with your real GitHub username/repo once you've pushed it.
GITHUB_RAW_URL = (
    "https://raw.githubusercontent.com/<you>/<repo>/main/summative/API/preprocessing.py"
)

USING_SHARED_MODULE = False
try:
    import urllib.request
    urllib.request.urlretrieve(GITHUB_RAW_URL, "preprocessing.py")
    from preprocessing import clean_raw_dataframe, build_feature_matrix
    USING_SHARED_MODULE = True
    print("Loaded preprocessing.py from GitHub -- this notebook and the API are now",
          "running the identical cleaning/encoding code.")
except Exception as e:
    print(f"Could not fetch preprocessing.py from GitHub ({e}).")
    print("Falling back to the local fallback functions defined in the cells below "
          "(kept identical to preprocessing.py by hand). Push this repo to GitHub, "
          "update GITHUB_RAW_URL above, and re-run this cell to use the shared module directly.")


## 1. Load the Raw Dataset & Profile What's Dirty

> Update `DATA_PATH` to wherever the CSV lives in your Colab session.


In [ ]:
DATA_PATH = "/content/global_freelancers_raw.csv"  # <-- UPDATE IF NEEDED

df_raw = pd.read_csv(DATA_PATH)
print(df_raw.shape)
df_raw.head(10)


In [ ]:
print('--- Missing values per column ---')
print(df_raw.isnull().sum())
print()
print('--- Inconsistent text values that need cleaning ---')
print('gender:', sorted(df_raw['gender'].unique()))
print('is_active:', sorted(df_raw['is_active'].dropna().astype(str).unique()))
print('hourly_rate (USD) sample formats:', df_raw['hourly_rate (USD)'].dropna().unique()[:10])
print('client_satisfaction sample formats:', df_raw['client_satisfaction'].dropna().unique()[:10])


**What's dirty (drives Section 1.1 cleaning code):**
- Missing values in `age`, `years_of_experience`, `hourly_rate (USD)` (the **target**), `rating`, 
  `is_active`, and `client_satisfaction`.
- `gender` has 10 inconsistent spellings (`f`, `F`, `Female`, `FEMALE`, `female`, `m`, `M`, `Male`, 
  `MALE`, `male`) that all mean only two categories.
- `is_active` mixes `1/0`, `Y/N`, `yes/no`, `True/False` for the same boolean concept.
- `hourly_rate (USD)` (target) mixes bare numbers, `"$40"`, and `"USD 40"` — all text (`object`) dtype.
- `client_satisfaction` mixes percentage strings (`"84%"`) with bare numbers (`"92"`).


### 1.1 Cleaning: normalize text, parse mixed formats, handle every NaN


In [ ]:
if USING_SHARED_MODULE:
    df = clean_raw_dataframe(df_raw)
else:
    # Local fallback -- kept IDENTICAL to preprocessing.clean_raw_dataframe()
    def clean_rate(x):
        if pd.isna(x):
            return np.nan
        s = str(x).replace('USD', '').replace('$', '').strip()
        try:
            return float(s)
        except ValueError:
            return np.nan

    def clean_gender(x):
        x = str(x).strip().lower()
        if x in ('m', 'male'):
            return 'male'
        if x in ('f', 'female'):
            return 'female'
        return np.nan

    active_map = {'1': 1, '0': 0, 'y': 1, 'n': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0}

    def clean_active(x):
        if pd.isna(x):
            return np.nan
        return active_map.get(str(x).strip().lower(), np.nan)

    def clean_pct(x):
        if pd.isna(x):
            return np.nan
        s = str(x).replace('%', '').strip()
        try:
            return float(s)
        except ValueError:
            return np.nan

    df = df_raw.copy()
    df['hourly_rate_usd'] = df['hourly_rate (USD)'].apply(clean_rate)
    df = df.dropna(subset=['hourly_rate_usd']).reset_index(drop=True)
    df['gender_clean'] = df['gender'].apply(clean_gender)
    df['is_active_clean'] = df['is_active'].apply(clean_active)
    df['client_satisfaction_clean'] = df['client_satisfaction'].apply(clean_pct)
    for col in ['age', 'years_of_experience', 'rating', 'client_satisfaction_clean']:
        df[col] = df[col].fillna(df[col].median())
    df['gender_clean'] = df['gender_clean'].fillna(df['gender_clean'].mode()[0])
    df['is_active_clean'] = df['is_active_clean'].fillna(df['is_active_clean'].mode()[0])

print('Remaining nulls in cleaned columns:')
print(df[['age', 'years_of_experience', 'rating', 'client_satisfaction_clean',
          'gender_clean', 'is_active_clean', 'hourly_rate_usd']].isnull().sum())


**Result:** every column used downstream is now free of NaNs, and `gender`, `is_active`, 
`client_satisfaction`, and the target `hourly_rate` are all fully numeric-ready or normalized to a 
small, consistent set of category labels.


## 2. Required Visualizations (inform feature engineering & modeling choices)


### 2.1 Feature & target distributions


In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
sns.histplot(df['hourly_rate_usd'], kde=False, ax=axes[0], color='steelblue', bins=6)
axes[0].set_title('Hourly Rate (USD)')
sns.histplot(df['years_of_experience'], kde=True, ax=axes[1], color='seagreen')
axes[1].set_title('Years of Experience')
sns.histplot(df['age'], kde=True, ax=axes[2], color='indianred')
axes[2].set_title('Age')
sns.histplot(df['rating'], kde=True, ax=axes[3], color='goldenrod')
axes[3].set_title('Rating')
plt.tight_layout()
plt.show()


**Interpretation:** `hourly_rate_usd` only takes 6 discrete values (20/30/40/50/75/100) rather than a 
smooth continuum — still a valid regression target, but it means the model is really learning a small 
set of pricing tiers. `years_of_experience`, `age`, and `rating` are all fairly evenly spread, so no 
extreme skew correction is needed beyond standard scaling.


### 2.2 Correlation heatmap


In [ ]:
plt.figure(figsize=(7, 5))
corr = df[['age', 'years_of_experience', 'rating', 'client_satisfaction_clean', 'hourly_rate_usd']].corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Heatmap (Numeric Features)')
plt.show()
print(corr['hourly_rate_usd'])


**Interpretation — directly changes expectations for modeling:** none of `age`, `years_of_experience`, 
`rating`, or `client_satisfaction` correlate meaningfully with `hourly_rate_usd` (all under ~0.1). No 
leakage column is found here (unlike a prior dataset we worked with), but this also means the model has 
a genuinely weak linear signal to learn from — a result the comparison table in Section 7 should be read 
in light of.


## 3. Feature Engineering

| Column | Decision | Reason |
|---|---|---|
| `freelancer_ID` | Drop | Identifier, no signal |
| `name` | Drop | Identifier / PII, no signal |
| `language` | Drop | Deterministic function of `country` (each country maps to exactly one language) — redundant |
| `gender` | Keep, clean to 2 categories, binary-encode | Normalized from 10 inconsistent spellings |
| `age` | Keep, numeric | No strong individual correlation, but kept as a standard demographic feature |
| `country` | Keep, one-hot | Captures regional pricing differences not carried by any other column |
| `primary_skill` | Keep, one-hot | Type of work is a plausible price driver |
| `years_of_experience` | Keep, numeric | Standard driver of freelance pricing |
| `rating` | Keep, numeric | Performance signal |
| `is_active` | Keep, binary (already 0/1 after cleaning) | Activity status may relate to pricing/availability |
| `client_satisfaction` | Keep, numeric (0-100 after cleaning) | Performance signal |


In [ ]:
if USING_SHARED_MODULE:
    df_model = build_feature_matrix(df)
else:
    # Local fallback -- kept IDENTICAL to preprocessing.build_feature_matrix()
    drop_cols = ['freelancer_ID', 'name', 'language', 'gender', 'is_active',
                 'client_satisfaction', 'hourly_rate (USD)']
    df_model = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df_model['gender_num'] = (df_model['gender_clean'] == 'male').astype(int)
    df_model = df_model.drop(columns=['gender_clean'])
    df_model = pd.get_dummies(df_model, columns=['country', 'primary_skill'], drop_first=True)

df_model.head()


## 4. Convert Remaining Categorical/Text Columns to Numeric


In [ ]:
print('Any non-numeric columns left?',
      df_model.drop(columns=['hourly_rate_usd']).select_dtypes(exclude=[np.number, bool]).columns.tolist())
print('Final shape:', df_model.shape)
df_model.head()


## 5. Train/Test Split & Standardization

Continuous columns are standardized with `StandardScaler`, fit only on the training split to avoid leakage.


In [ ]:
target_col = 'hourly_rate_usd'
feature_cols = [c for c in df_model.columns if c != target_col]

X = df_model[feature_cols]
y = df_model[target_col].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

cols_to_scale = ['age', 'years_of_experience', 'rating', 'client_satisfaction_clean']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[cols_to_scale] = scaler.fit_transform(X_train[cols_to_scale])
X_test_scaled[cols_to_scale] = scaler.transform(X_test[cols_to_scale])

# Numpy views used by the manual gradient-descent training loops below
Xtr = X_train_scaled.values.astype(float)
ytr = y_train
Xte = X_test_scaled.values.astype(float)
yte = y_test

X_train_scaled.head()


## 6. Four Regression Models

Two **linear regression** implementations (both built on scikit-learn's `SGDRegressor`, both trained 
with gradient descent, differing in how many samples are used per parameter update), plus a **tree** 
model and an **ensemble** model:

- **Model A — Stochastic Gradient Descent (SGD) Linear Regression:** one training example per weight update.
- **Model B — Mini-Batch Gradient Descent Linear Regression:** a batch of 32 examples per weight update.
- **Model C — Decision Tree Regressor** (tree algorithm).
- **Model D — Random Forest Regressor** (ensemble algorithm).


### 6.1 Model A — Stochastic Gradient Descent Linear Regression (batch size = 1)


In [ ]:
n_epochs_a = 15

model_a = SGDRegressor(
    loss='squared_error', penalty='l2', alpha=0.0001,
    learning_rate='invscaling', eta0=0.01,
    max_iter=1, tol=None, warm_start=True, random_state=RANDOM_STATE,
)

train_losses_a, test_losses_a = [], []
n = len(Xtr)

for epoch in range(n_epochs_a):
    order = rng.permutation(n)
    for i in order:
        model_a.partial_fit(Xtr[i:i+1], ytr[i:i+1])   # one sample = one update
    train_losses_a.append(mean_squared_error(ytr, model_a.predict(Xtr)))
    test_losses_a.append(mean_squared_error(yte, model_a.predict(Xte)))

print('Model A final train MSE:', train_losses_a[-1])
print('Model A final test MSE:', test_losses_a[-1])


### 6.2 Model B — Mini-Batch Gradient Descent Linear Regression (batch size = 32)


In [ ]:
n_epochs_b = 60
batch_size = 32

model_b = SGDRegressor(
    loss='squared_error', penalty='l2', alpha=0.0001,
    learning_rate='invscaling', eta0=0.01,
    max_iter=1, tol=None, warm_start=True, random_state=RANDOM_STATE,
)

train_losses_b, test_losses_b = [], []

for epoch in range(n_epochs_b):
    order = rng.permutation(n)
    for start in range(0, n, batch_size):
        batch_idx = order[start:start + batch_size]
        model_b.partial_fit(Xtr[batch_idx], ytr[batch_idx])   # 32 samples = one update
    train_losses_b.append(mean_squared_error(ytr, model_b.predict(Xtr)))
    test_losses_b.append(mean_squared_error(yte, model_b.predict(Xte)))

print('Model B final train MSE:', train_losses_b[-1])
print('Model B final test MSE:', test_losses_b[-1])


### 6.3 Loss curves — Model A vs Model B


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(train_losses_a, label='Train', color='steelblue')
axes[0].plot(test_losses_a, label='Test', color='darkorange')
axes[0].set_title('Model A (Stochastic GD) Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE'); axes[0].legend()

axes[1].plot(train_losses_b, label='Train', color='steelblue')
axes[1].plot(test_losses_b, label='Test', color='darkorange')
axes[1].set_title('Model B (Mini-Batch GD) Loss')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('MSE'); axes[1].legend()

plt.tight_layout()
plt.show()


**Interpretation:** both curves flatten quickly since there is little linear signal to fit (Section 
2.2); train and test loss stay close, so the flat loss reflects weak features rather than overfitting.


### 6.4 Model C — Decision Tree Regressor & Model D — Random Forest Regressor


In [ ]:
model_c = DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=RANDOM_STATE)
model_c.fit(Xtr, ytr)

model_d = RandomForestRegressor(n_estimators=300, max_depth=6, random_state=RANDOM_STATE, n_jobs=-1)
model_d.fit(Xtr, ytr)

print('Decision Tree and Random Forest trained.')


### 6.5 Comparing all 4 models


In [ ]:
def evaluate(name, y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    return {'Model': name, 'MSE': mse, 'RMSE': np.sqrt(mse),
            'MAE': mean_absolute_error(y_true, y_pred), 'R2': r2_score(y_true, y_pred)}

results = [
    evaluate('Model A - Stochastic GD Linear Regression', yte, model_a.predict(Xte)),
    evaluate('Model B - Mini-Batch GD Linear Regression', yte, model_b.predict(Xte)),
    evaluate('Model C - Decision Tree Regressor', yte, model_c.predict(Xte)),
    evaluate('Model D - Random Forest Regressor', yte, model_d.predict(Xte)),
]

results_df = pd.DataFrame(results).sort_values('RMSE').reset_index(drop=True)
results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(x='Model', y='RMSE', data=results_df, ax=axes[0], palette='viridis')
axes[0].set_title('RMSE by Model (lower = less loss)')
axes[0].tick_params(axis='x', rotation=25)
sns.barplot(x='Model', y='R2', data=results_df, ax=axes[1], palette='viridis')
axes[1].set_title('R² by Model (higher = better fit)')
axes[1].tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.show()


**Interpretation:** given the near-zero correlations found in Section 2.2, expect R² values close to 
(or even below) zero for all four models — that is an honest reflection of this dataset having very 
little real predictive signal for `hourly_rate`, not a coding error. The comparison and the 'least loss' 
selection below are still valid and required steps regardless of how strong the signal turns out to be.


### 6.6 Criteria for choosing the best model

The **best-performing model is the one with the lowest test-set RMSE (equivalently the lowest MSE / 
least loss, and the highest R²)** on data the model never trained on. RMSE is used as the primary tie-
breaker because it is in the same units as `hourly_rate_usd` (dollars), making the error directly 
interpretable; R² is reported as a secondary, scale-free confirmation of the same ranking.


In [ ]:
model_lookup = {
    'Model A - Stochastic GD Linear Regression': model_a,
    'Model B - Mini-Batch GD Linear Regression': model_b,
    'Model C - Decision Tree Regressor': model_c,
    'Model D - Random Forest Regressor': model_d,
}

best_model_name = results_df.iloc[0]['Model']   # lowest RMSE = least loss
best_model = model_lookup[best_model_name]
print(f'Best performing model (least loss): {best_model_name}')
print(results_df.iloc[0])

joblib.dump(best_model, 'best_rate_model.pkl')
joblib.dump(scaler, 'feature_scaler.pkl')
joblib.dump(list(X_train_scaled.columns), 'feature_columns.pkl')
joblib.dump(cols_to_scale, 'columns_to_scale.pkl')
joblib.dump({c: df[c].median() for c in ['age', 'years_of_experience', 'rating', 'client_satisfaction_clean']}, 'train_medians.pkl')
print('Saved best_rate_model.pkl and supporting artifacts.')


## 7. Scatter Plot — Best-Fit Line of the (Linear Regression) Model

The trained linear model uses many features, so we visualize its fit along `years_of_experience` by 
plotting that feature's coefficient and intercept from the trained model as a line over the actual data 
(other features held at their standardized mean of 0).


In [ ]:
rmse_a = results_df.set_index('Model').loc['Model A - Stochastic GD Linear Regression', 'RMSE']
rmse_b = results_df.set_index('Model').loc['Model B - Mini-Batch GD Linear Regression', 'RMSE']
linear_model_for_plot = model_b if rmse_b <= rmse_a else model_a

feat_idx = list(X_train_scaled.columns).index('years_of_experience')
coef = linear_model_for_plot.coef_[feat_idx]
intercept = linear_model_for_plot.intercept_[0]

x_vals = X_train_scaled['years_of_experience'].values
x_line = np.linspace(x_vals.min(), x_vals.max(), 100)
y_line = intercept + coef * x_line

plt.figure(figsize=(8, 5))
plt.scatter(x_vals, y_train, alpha=0.4, color='steelblue', label='Training data')
plt.plot(x_line, y_line, color='red', linewidth=2, label='Fitted line (years_of_experience slice)')
plt.xlabel('Years of Experience (standardized)')
plt.ylabel('Hourly Rate (USD)')
plt.title('Best-Fit Line — Linear Regression Model')
plt.legend()
plt.show()


**Interpretation:** consistent with the weak correlation in Section 2.2, the line is close to flat — a 
faithful reflection of the trained model rather than a plotting issue.


## 8. Prediction on a Single Row from the Test Set


In [ ]:
test_row_index = 0
single_row = X_test_scaled.iloc[[test_row_index]]
true_value = y_test[test_row_index]
predicted_value = best_model.predict(single_row)[0]

print(f'True hourly rate:      ${true_value:,.2f}')
print(f'Predicted hourly rate: ${predicted_value:,.2f}')
print(f'Absolute error:        ${abs(true_value - predicted_value):,.2f}')


## 9. Prediction Script for a New Freelancer Profile (Task 2)


In [ ]:
import joblib
import pandas as pd

loaded_model = joblib.load('best_rate_model.pkl')
loaded_scaler = joblib.load('feature_scaler.pkl')
loaded_feature_cols = joblib.load('feature_columns.pkl')
loaded_cols_to_scale = joblib.load('columns_to_scale.pkl')
loaded_train_medians = joblib.load('train_medians.pkl')


def predict_hourly_rate(new_record: dict) -> float:
    """
    new_record example:
    {
        'gender': 'female',            # any messy variant is fine, e.g. 'F', 'FEMALE'
        'age': 34,
        'country': 'India',
        'primary_skill': 'Machine Learning',
        'years_of_experience': 6,
        'rating': 4.2,
        'is_active': 'yes',            # any messy variant is fine, e.g. 1, 'Y', True
        'client_satisfaction': '88%',  # or a bare number like 88
    }
    Returns the predicted hourly rate in USD (float). Missing keys fall back to training medians.
    """
    rec = new_record.copy()
    row = {col: 0 for col in loaded_feature_cols}

    row['age'] = rec.get('age', loaded_train_medians['age'])
    row['years_of_experience'] = rec.get('years_of_experience', loaded_train_medians['years_of_experience'])
    row['rating'] = rec.get('rating', loaded_train_medians['rating'])

    gender_str = str(rec.get('gender', 'male')).strip().lower()
    row['gender_num'] = 1 if gender_str in ('m', 'male') else 0

    active_map = {'1': 1, '0': 0, 'y': 1, 'n': 0, 'yes': 1, 'no': 0, 'true': 1, 'false': 0}
    row['is_active_clean'] = active_map.get(str(rec.get('is_active', 1)).strip().lower(), 1)

    sat_raw = rec.get('client_satisfaction', None)
    if sat_raw is None:
        row['client_satisfaction_clean'] = loaded_train_medians['client_satisfaction_clean']
    else:
        row['client_satisfaction_clean'] = float(str(sat_raw).replace('%', '').strip())

    country_col = f"country_{rec['country']}"
    if country_col in row:
        row[country_col] = 1

    skill_col = f"primary_skill_{rec['primary_skill']}"
    if skill_col in row:
        row[skill_col] = 1

    row_df = pd.DataFrame([row])[loaded_feature_cols]
    row_df[loaded_cols_to_scale] = loaded_scaler.transform(row_df[loaded_cols_to_scale])
    return loaded_model.predict(row_df)[0]


sample_freelancer = {
    'gender': 'female',
    'age': 34,
    'country': 'India',
    'primary_skill': 'Machine Learning',
    'years_of_experience': 6,
    'rating': 4.2,
    'is_active': 'yes',
    'client_satisfaction': '88%',
}

predicted_rate = predict_hourly_rate(sample_freelancer)
print(f'Predicted hourly rate: ${predicted_rate:,.2f}')


## 10. Summary

- **Mission/data:** non-generic hourly-rate prediction on a rich, genuinely raw/messy dataset (1,000 
  rows; 21 countries, 10 skills, mixed-format target and boolean/percentage columns).
- **Cleaning:** parsed the `$`/`USD`-formatted target, normalized 10 gender spellings to 2 categories, 
  normalized `is_active` from 8 different boolean encodings to a clean 0/1, parsed `client_satisfaction` 
  from mixed `%`/bare-number text, dropped rows with a missing target, and median/mode-imputed all 
  remaining missing feature values — the dataset has **zero NaNs and zero non-numeric training columns** 
  after Section 1 and Section 4.
- **Visualizations that shaped modeling:** the correlation heatmap showed **no strong linear driver** of 
  `hourly_rate` among the numeric features — an honest, dataset-driven finding carried through to the 
  model interpretation rather than hidden.
- **4 models compared:** Stochastic GD Linear Regression, Mini-Batch GD Linear Regression, Decision Tree 
  Regressor, Random Forest Regressor.
- **Best-model criteria:** lowest test-set RMSE (least loss), confirmed by R².
- **Saved:** best model + scaler + feature schema + training medians (`best_rate_model.pkl` and supporting files).
- **Verified:** prediction on a real test-set row, plus a reusable `predict_hourly_rate()` function that 
  accepts messy real-world input formats and cleans them the same way as training.
